<a href="https://colab.research.google.com/github/abhi6378/AI-Intelligent-Email-Triage-Agent/blob/main/Document_AI.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:

import kagglehub
kagglehub.login()


Kaggle credentials set.
Kaggle credentials successfully validated.


In [3]:
# IMPORTANT: RUN THIS CELL IN ORDER TO IMPORT YOUR KAGGLE DATA SOURCES,
# THEN FEEL FREE TO DELETE THIS CELL.
# NOTE: THIS NOTEBOOK ENVIRONMENT DIFFERS FROM KAGGLE'S PYTHON
# ENVIRONMENT SO THERE MAY BE MISSING LIBRARIES USED BY YOUR
# NOTEBOOK.

abhishekswami10_document_ai_offline_models_path = kagglehub.dataset_download('abhishekswami10/document-ai-offline-models')

print('Data source import complete.')


Data source import complete.


In [ ]:
import os
from pathlib import Path

# ── Kaggle Dataset Mount Root ─────────────────────────────────────────────────
# Your dataset is mounted here at notebook boot — zero download time.
# Verify subfolders with:  !ls /kaggle/input/document-ai-offline-models/document-ai-models/
BASE = (
    "/kaggle/input/datasets/abhishekswami10/document-ai-offline-models"
    "/document-ai-models"
)

# ── Individual Model Paths ────────────────────────────────────────────────────
#     Run !ls /kaggle/input/  to confirm the dataset folder name.
MODEL_PATHS = {
    # Pre-quantized AWQ: ~5 GB on disk, ~5-6 GB VRAM, loads in ~10s
    # ✅ No bitsandbytes   ✅ No load_in_4bit   ✅ autoawq via quantization_config
    "qwen"     : f"{BASE}/qwen2.5-vl-7b-awq",

    # Donut: encoder-decoder, no internal OCR engine, end-to-end doc parsing
    "donut"    : f"naver-clova-ix/donut-base-finetuned-cord-v2",

    # Florence-2: unified VLM with native <OCR> and <OCR_WITH_REGION> tasks
    "florence" : f"{BASE}/florence-2-large",

    # TrOCR: Transformer OCR fine-tuned on IAM handwriting dataset
    "trocr"    : f"{BASE}/trocr-handwritten",

    # PaddleOCR: auto-downloads its ~50 MB det/rec weights on first call
    # Set a path below if you have cached weights in your dataset:
    # "paddle" : f"{BASE}/paddleocr-weights"
    "paddle"   : None,

    # DocTR: auto-downloads ~200 MB weights on first call
    # Set a path below if cached:
    # "doctr"  : f"{BASE}/doctr-weights"
    "doctr"    : None,
}

# ── Use-Case → Model Map ──────────────────────────────────────────────────────
# Single source of truth — drives both the Gradio dropdown AND the Router.
# String values must EXACTLY match the ROUTER keys in Cell 11.
USE_CASE_MODEL_MAP = {
    "Receipt"     : ["PaddleOCR", "Qwen2.5-VL-7B-AWQ", "Donut"],
    "Invoice"     : ["Qwen2.5-VL-7B-AWQ", "Donut", "Florence-2-Large"],
    "Handwriting" : ["TrOCR-Large-Handwritten", "Qwen2.5-VL-7B-AWQ"],
    "Legal PDF"   : ["DocTR", "PaddleOCR", "Qwen2.5-VL-7B-AWQ"],
}

# ── Working Directory Paths ───────────────────────────────────────────────────
WORK_DIR    = Path("/kaggle/working")
OUTPUT_FILE = WORK_DIR / "model_output.txt"   # Every script writes here
UPLOAD_IMG  = WORK_DIR / "uploaded_image.jpg" # Gradio saves upload here

# ── venv Python Binaries ──────────────────────────────────────────────────────
PADDLE_PY = "/kaggle/working/paddle_env/bin/python"
DOCTR_PY  = "/kaggle/working/doctr_env/bin/python"
HF_PY     = "/kaggle/working/hf_env/bin/python"

# ── Startup Validation ────────────────────────────────────────────────────────
print("=" * 60)
print("  📁  MODEL PATH VALIDATION")
print("=" * 60)
for name, path in MODEL_PATHS.items():
    if path is None:
        print(f"  ⚠️  {name:<12} → auto-download at first run")
    elif Path(path).exists():
        print(f"  ✅  {name:<12} → {path}")
    else:
        print(f"  ❌  {name:<12} → NOT FOUND: {path}")
        print(f"            Run: !ls {BASE}")
print("=" * 60)
print("✅  Cell 1 complete — config loaded")

  📁  MODEL PATH VALIDATION
  ✅  qwen         → /kaggle/input/datasets/abhishekswami10/document-ai-offline-models/document-ai-models/qwen2.5-vl-7b-awq
  ❌  donut        → NOT FOUND: naver-clova-ix/donut-base-finetuned-cord-v2
            Run: !ls /kaggle/input/datasets/abhishekswami10/document-ai-offline-models/document-ai-models
  ✅  florence     → /kaggle/input/datasets/abhishekswami10/document-ai-offline-models/document-ai-models/florence-2-large
  ✅  trocr        → /kaggle/input/datasets/abhishekswami10/document-ai-offline-models/document-ai-models/trocr-handwritten
  ⚠️  paddle       → auto-download at first run
  ⚠️  doctr        → auto-download at first run
✅  Cell 1 complete — config loaded


In [ ]:
%%bash
pip install --quiet virtualenv

# --system-site-packages: inherits wrapt + other Kaggle system packages
virtualenv --system-site-packages /kaggle/working/paddle_env
echo "✅ paddle_env created"

/kaggle/working/paddle_env/bin/pip install --quiet --upgrade pip

/kaggle/working/paddle_env/bin/pip install --quiet \
    paddlepaddle==2.6.2 \
    "paddleocr>=2.7.0,<2.8" \
    "opencv-python-headless==4.8.1.78" \
    "numpy<2.0" \
    Pillow \
    requests

echo "✅ paddle_env — PaddleOCR installed"
/kaggle/working/paddle_env/bin/python -c "from paddleocr import PaddleOCR; print('  import OK')"

created virtual environment CPython3.12.12.final.0-64-x86_64 in 265ms
  creator CPython3Posix(dest=/kaggle/working/paddle_env, clear=False, no_vcs_ignore=False, global=True)
  seeder FromAppData(download=False, pip=bundle, via=copy, app_data_dir=/root/.cache/virtualenv)
    added seed packages: astor==0.8.1, attrdict==2.0.1, bce_python_sdk==0.9.64, cssselect==1.4.0, cssutils==2.11.1, fire==0.7.1, flask_babel==4.0.0, imgaug==0.4.0, lmdb==2.0.0, numpy==1.26.4, opencv_contrib_python==4.6.0.66, opencv_python==4.6.0.66, opencv_python_headless==4.8.1.78, opt_einsum==3.3.0, paddleocr==2.7.3, paddlepaddle==2.6.2, pdf2docx==0.5.12, pip==26.0.1, premailer==3.10.0, pymupdf==1.27.2, python_docx==1.2.0, rapidfuzz==3.14.3, rarfile==4.2, visualdl==2.5.3
  activators BashActivator,CShellActivator,FishActivator,NushellActivator,PowerShellActivator,PythonActivator
✅ paddle_env created
✅ paddle_env — PaddleOCR installed
  import OK


In [ ]:
%%bash
# ── Create doctr_env ──────────────────────────────────────────────────────────
# python-doctr now defaults to PyTorch; the [torch] extra is deprecated.
# Must be isolated from paddle_env (conflicting OpenCV builds)
# and hf_env (different torch version requirements).

virtualenv --system-site-packages /kaggle/working/doctr_env
echo "✅ doctr_env created"

/kaggle/working/doctr_env/bin/pip install --quiet --upgrade pip

/kaggle/working/doctr_env/bin/pip install --quiet \
    python-doctr \
    torch \
    torchvision \
    Pillow \
    PyMuPDF \
    pdf2image

echo "✅ doctr_env — DocTR installed"
/kaggle/working/doctr_env/bin/python -c "from doctr.models import ocr_predictor; print('  import OK')"

created virtual environment CPython3.12.12.final.0-64-x86_64 in 259ms
  creator CPython3Posix(dest=/kaggle/working/doctr_env, clear=False, no_vcs_ignore=False, global=True)
  seeder FromAppData(download=False, pip=bundle, via=copy, app_data_dir=/root/.cache/virtualenv)
    added seed packages: anyascii==0.3.3, langdetect==1.0.9, pip==26.0.1, pymupdf==1.27.2, pypdfium2==5.6.0, python_doctr==1.0.1, rapidfuzz==3.14.3, validators==0.35.0
  activators BashActivator,CShellActivator,FishActivator,NushellActivator,PowerShellActivator,PythonActivator
✅ doctr_env created
✅ doctr_env — DocTR installed
  import OK


In [ ]:
%%bash
# Remove old broken hf_env
rm -rf /kaggle/working/hf_env

# Step 1: autoawq into base kernel (already done, skip if done)
pip install --quiet autoawq
echo "✅ autoawq in base kernel"

# Step 2: Create fresh hf_env
virtualenv --system-site-packages /kaggle/working/hf_env
echo "✅ hf_env created"

# Step 3: Install with transformers==4.51.3 (matches autoawq 0.2.9 requirement)
/kaggle/working/hf_env/bin/pip install --quiet --upgrade pip

/kaggle/working/hf_env/bin/pip install --quiet \
    transformers==4.51.3 \
    torch \
    torchvision \
    accelerate \
    einops \
    timm \
    sentencepiece \
    qwen-vl-utils \
    Pillow \
    PyMuPDF \
    pdf2image \
    numpy

echo "✅ hf_env — ecosystem installed"

# Step 4: Verify — we NEVER import awq directly, transformers handles it internally
/kaggle/working/hf_env/bin/python -c "
import transformers
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
print(f'  transformers : {transformers.__version__}')
print('  ✅ Qwen2.5-VL import OK')
print('  ✅ AWQ backend will be auto-detected at model load time')
"

✅ autoawq in base kernel
created virtual environment CPython3.12.12.final.0-64-x86_64 in 180ms
  creator CPython3Posix(dest=/kaggle/working/hf_env, clear=False, no_vcs_ignore=False, global=True)
  seeder FromAppData(download=False, pip=bundle, via=copy, app_data_dir=/root/.cache/virtualenv)
    added seed packages: pip==26.0.1
  activators BashActivator,CShellActivator,FishActivator,NushellActivator,PowerShellActivator,PythonActivator
✅ hf_env created
✅ hf_env — ecosystem installed
  transformers : 4.51.3
  ✅ Qwen2.5-VL import OK
  ✅ AWQ backend will be auto-detected at model load time


2026-03-19 04:51:12.272584: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773895872.294402     267 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1773895872.300950     267 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1773895872.317863     267 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773895872.317887     267 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1773895872.317890     267 computation_placer.cc:177] computation placer alr

## Paddle Ocr

In [ ]:

%%writefile /kaggle/working/run_paddle.py
# run_paddle.py — PaddleOCR inference subprocess
# Runs inside: paddle_env
# Usage: paddle_env/bin/python run_paddle.py <image_path> <use_case> <output_file>
# On exit: OS reclaims all VRAM/RAM — no manual cleanup needed.

import sys
from pathlib import Path
from paddleocr import PaddleOCR

# ── Args ──────────────────────────────────────────────────────────────────────
image_path  = sys.argv[1]   # Absolute path to image (or PDF-page JPEG)
use_case    = sys.argv[2]   # "Receipt" / "Legal PDF"
output_file = sys.argv[3]   # Where to write extracted text

print(f"[PaddleOCR] image={image_path}  use_case={use_case}")

# ── Engine ────────────────────────────────────────────────────────────────────
# use_angle_cls=True : corrects 90°/180°-rotated text (common in scanned docs)
# lang="en"          : English model (change to "ch" for Chinese docs)
# show_log=False     : silences PaddlePaddle verbose init logs
ocr = PaddleOCR(use_angle_cls=True, lang="en", show_log=False)

# ── Inference ─────────────────────────────────────────────────────────────────
results = ocr.ocr(image_path, cls=True)

# ── Parse ─────────────────────────────────────────────────────────────────────
# Structure: [ page [ detection [ bbox, (text, confidence) ] ] ]
lines = []
for page in (results or []):
    if not page:
        continue
    for det in page:
        text = det[1][0]
        conf = det[1][1]
        lines.append(f"{text}  [conf:{conf:.3f}]")

out = "\n".join(lines) if lines else "[PaddleOCR] No text detected."

# ── Write output ──────────────────────────────────────────────────────────────
Path(output_file).write_text(out, encoding="utf-8")
print(f"[PaddleOCR] Done — {len(lines)} lines extracted.")
print(out[:300])



Overwriting /kaggle/working/run_paddle.py


## Doctr

In [ ]:

%%writefile /kaggle/working/run_doctr.py
# run_doctr.py — DocTR OCR inference subprocess
# Runs inside: doctr_env
# Usage: doctr_env/bin/python run_doctr.py <image_path> <use_case> <output_file>
# DocTR pipeline: db_resnet50 (detection) → crnn_vgg16_bn (recognition)

import sys
import os
from pathlib import Path

# Force PyTorch backend before any doctr import
os.environ["USE_TORCH"] = "1"
# Cache weights inside /kaggle/working so they persist across reruns
os.environ["DOCTR_CACHE_DIR"] = "/kaggle/working/doctr_cache"

from doctr.io import DocumentFile
from doctr.models import ocr_predictor

# ── Args ──────────────────────────────────────────────────────────────────────
image_path  = sys.argv[1]
use_case    = sys.argv[2]
output_file = sys.argv[3]

print(f"[DocTR] image={image_path}  use_case={use_case}")

# ── Load pipeline ─────────────────────────────────────────────────────────────
# pretrained=True downloads weights once, then serves from DOCTR_CACHE_DIR.
# db_resnet50     : best general-purpose text region detector
# crnn_vgg16_bn   : strong printed-text recogniser, good on legal docs
model = ocr_predictor(
    det_arch="db_resnet50",
    reco_arch="crnn_vgg16_bn",
    pretrained=True
)

# ── Load document ─────────────────────────────────────────────────────────────
# DocumentFile handles both images and PDFs natively
ext = Path(image_path).suffix.lower()
doc = DocumentFile.from_pdf(image_path) if ext == ".pdf" else DocumentFile.from_images([image_path])

# ── Inference ─────────────────────────────────────────────────────────────────
result = model(doc)

# ── Render ────────────────────────────────────────────────────────────────────
# .render() reconstructs spatially-ordered plain text preserving paragraphs
out = result.render()
out = out.strip() if out.strip() else "[DocTR] No text detected."

# ── Write output ──────────────────────────────────────────────────────────────
Path(output_file).write_text(out, encoding="utf-8")
print(f"[DocTR] Done — output written.")
print(out[:300])


Overwriting /kaggle/working/run_doctr.py


## Qwen LLm

In [ ]:
%%writefile /kaggle/working/run_qwen.py
import sys
import time
import warnings
import torch
from pathlib import Path
from PIL import Image

warnings.filterwarnings("ignore")

from transformers import AutoConfig, Qwen2_5_VLForConditionalGeneration, AutoProcessor

image_path  = sys.argv[1]
use_case    = sys.argv[2]
output_file = sys.argv[3]
model_path  = sys.argv[4]
prompt_text = sys.argv[5]  # CHANGED: Now receives the custom prompt from the UI

print(f"[Qwen2.5-AWQ] image={image_path}")
print(f"[Qwen2.5-AWQ] use_case={use_case}")
print(f"[Qwen2.5-AWQ] prompt={prompt_text}")

print("[Qwen2.5-AWQ] Loading processor...")
processor = AutoProcessor.from_pretrained(model_path, use_fast=True, trust_remote_code=True)

print("[Qwen2.5-AWQ] Patching AWQ Config & Loading model weights...")
config = AutoConfig.from_pretrained(model_path, trust_remote_code=True)

if hasattr(config, "quantization_config"):
    ignores = config.quantization_config.get("modules_to_not_convert", [])
    if "lm_head" not in ignores:
        ignores.append("lm_head")
    config.quantization_config["modules_to_not_convert"] = ignores

t1 = time.time()
model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    model_path,
    config=config,
    torch_dtype=torch.float16,
    device_map="auto",
    trust_remote_code=True,
)
model.eval()
print(f"[Qwen2.5-AWQ] Model loaded ({time.time()-t1:.1f}s)")

image = Image.open(image_path).convert("RGB")
messages = [{"role": "user", "content": [{"type": "image", "image": image}, {"type": "text", "text": prompt_text}]}]
text_prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)

try:
    from qwen_vl_utils import process_vision_info
    image_inputs, video_inputs = process_vision_info(messages)
    inputs = processor(text=[text_prompt], images=image_inputs, videos=video_inputs, return_tensors="pt", padding=True).to("cuda")
except ImportError:
    inputs = processor(text=[text_prompt], images=[image], return_tensors="pt", padding=True).to("cuda")

print("[Qwen2.5-AWQ] Running inference...")
with torch.no_grad():
    generated_ids = model.generate(
        **inputs,
        max_new_tokens=1024,
        do_sample=False,
        temperature=None,
        top_p=None,
    )

prompt_len = inputs["input_ids"].shape[1]
output_ids = generated_ids[:, prompt_len:]
out = processor.batch_decode(output_ids, skip_special_tokens=True, clean_up_tokenization_spaces=True)[0].strip()

del model, inputs, generated_ids
torch.cuda.empty_cache()

Path(output_file).write_text(out or "[Qwen2.5-AWQ] No output generated.", encoding="utf-8")
print(f"[Qwen2.5-AWQ] Done — output written.")

Overwriting /kaggle/working/run_qwen.py


## Donut

In [ ]:
%%writefile /kaggle/working/run_donut.py
import sys
import re
import json
import torch
from pathlib import Path
from PIL import Image
from transformers import DonutProcessor, VisionEncoderDecoderModel

image_path  = sys.argv[1]
use_case    = sys.argv[2]
output_file = sys.argv[3]
model_path  = sys.argv[4]

print(f"[Donut] image={image_path}  use_case={use_case}")
print("[Donut] Loading processor and model...")

processor = DonutProcessor.from_pretrained(model_path)
model     = VisionEncoderDecoderModel.from_pretrained(
    model_path,
    torch_dtype=torch.float32,
)
device = "cuda:0" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()

# FIXED: Dynamically extract the correct prompt token from the model config
if hasattr(model.config, "task_start_token"):
    task_prompt = model.config.task_start_token
else:
    task_prompt = "<s_cord-v2>"
print(f"[Donut] Using Task prompt: {task_prompt}")

image = Image.open(image_path).convert("RGB")
pixel_values = processor(image, return_tensors="pt").pixel_values.to(device, torch.float32)

decoder_input_ids = processor.tokenizer(task_prompt, add_special_tokens=False, return_tensors="pt").input_ids.to(device)

print("[Donut] Running inference with Beam Search...")
with torch.no_grad():
    outputs = model.generate(
        pixel_values,
        decoder_input_ids=decoder_input_ids,
        max_length=1536,
        pad_token_id=processor.tokenizer.pad_token_id,
        eos_token_id=processor.tokenizer.eos_token_id,
        use_cache=True,
        num_beams=3,             # FIXED: Beam search stops Donut from instantly outputting EOS
        early_stopping=True,
        bad_words_ids=[[processor.tokenizer.unk_token_id]],
        return_dict_in_generate=True,
    )

sequence = processor.batch_decode(outputs.sequences)[0]
sequence = sequence.replace(processor.tokenizer.eos_token, "").replace(processor.tokenizer.pad_token, "")
sequence = re.sub(r"<.*?>", "", sequence, count=1).strip()

try:
    parsed = processor.token2json(sequence)
    out    = json.dumps(parsed, indent=2, ensure_ascii=False)
except Exception:
    out = sequence

del model, pixel_values, outputs
torch.cuda.empty_cache()

Path(output_file).write_text(out or "[Donut] No output generated.", encoding="utf-8")
print(f"[Donut] Done — output written.")

Writing /kaggle/working/run_donut.py


## Florence - 2

In [ ]:
%%writefile /kaggle/working/run_florence.py
import sys
import json
import torch
from pathlib import Path
from PIL import Image
from transformers import AutoProcessor, AutoModelForCausalLM

image_path  = sys.argv[1]
use_case    = sys.argv[2]
output_file = sys.argv[3]
model_path  = sys.argv[4]

print(f"[Florence-2] image={image_path}  use_case={use_case}")

TASK_MAP = {
    "Invoice" : "<OCR_WITH_REGION>",
    "default" : "<OCR>",
}
task = TASK_MAP.get(use_case, TASK_MAP["default"])
print(f"[Florence-2] Task: {task}")

print("[Florence-2] Loading model from local path...")
processor = AutoProcessor.from_pretrained(model_path, trust_remote_code=True)
model     = AutoModelForCausalLM.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
    device_map="cuda:0",      # FIXED: Force onto a single GPU to prevent multi-GPU tensor crashes
    trust_remote_code=True,
)
model.eval()
print("[Florence-2] Model ready.")

image  = Image.open(image_path).convert("RGB")
inputs = processor(text=task, images=image, return_tensors="pt").to("cuda:0", torch.float16)

print("[Florence-2] Running inference...")
with torch.no_grad():
    generated_ids = model.generate(
        input_ids=inputs["input_ids"],
        pixel_values=inputs["pixel_values"],
        max_new_tokens=1024,
        do_sample=False,
        num_beams=3,
        early_stopping=True,
    )

raw    = processor.batch_decode(generated_ids, skip_special_tokens=False)[0]
parsed = processor.post_process_generation(
    raw, task=task, image_size=(image.width, image.height)
)

result = parsed.get(task, "")
if isinstance(result, str):
    out = result
else:
    labels = result.get("labels", [])
    out    = "\n".join(labels) if labels else json.dumps(result, indent=2)

del model, inputs, generated_ids
torch.cuda.empty_cache()

Path(output_file).write_text(out or "[Florence-2] No output generated.", encoding="utf-8")
print(f"[Florence-2] Done — output written.")

Writing /kaggle/working/run_florence.py


## Tr-Ocr

In [ ]:

%%writefile /kaggle/working/run_trocr.py
# run_trocr.py — TrOCR-Large-Handwritten inference subprocess
# Runs inside: hf_env
# Usage: hf_env/bin/python run_trocr.py <image_path> <use_case> <output_file> <model_path>
#
# IMPORTANT: TrOCR is a single-line model.
# For a full-page image we MUST segment it into text lines first.
# This script uses a horizontal-projection segmenter before running TrOCR.

import sys
import torch
import numpy as np
from pathlib import Path
from PIL import Image, ImageOps
from transformers import TrOCRProcessor, VisionEncoderDecoderModel

# ── Args ──────────────────────────────────────────────────────────────────────
image_path  = sys.argv[1]
use_case    = sys.argv[2]
output_file = sys.argv[3]
model_path  = sys.argv[4]

print(f"[TrOCR] image={image_path}  use_case={use_case}")

# ── Line segmenter ────────────────────────────────────────────────────────────
def segment_lines(image: Image.Image, min_height: int = 10) -> list:
    '''
    Horizontal-projection line segmentation.
    Sums dark pixels per row, groups consecutive text rows into crops.

    Args:
        image      : PIL RGB Image
        min_height : Minimum row height (px) to be treated as a valid line

    Returns:
        List of PIL Image crops, one per text line.
        Falls back to [original_image] if no lines are detected.
    '''
    gray    = ImageOps.grayscale(image)
    arr     = np.array(gray, dtype=np.float32)

    # Normalise: ensure background is bright, text is dark
    if arr.mean() < 128:
        arr = 255.0 - arr

    # Horizontal projection: high sum → text row
    row_sums  = (255.0 - arr).sum(axis=1)
    threshold = row_sums.max() * 0.05   # 5% of peak row density = text

    in_line    = False
    line_start = 0
    crops      = []

    for idx, val in enumerate(row_sums):
        if not in_line and val > threshold:
            in_line = True
            line_start = idx
        elif in_line and val <= threshold:
            in_line = False
            if (idx - line_start) >= min_height:
                crops.append(image.crop((0, line_start, image.width, idx)))

    # Handle text touching the bottom edge
    if in_line and (len(row_sums) - line_start) >= min_height:
        crops.append(image.crop((0, line_start, image.width, len(row_sums))))

    return crops if crops else [image]   # fallback: whole image as one line


# ── Load ──────────────────────────────────────────────────────────────────────
print("[TrOCR] Loading processor and model...")
processor = TrOCRProcessor.from_pretrained(model_path)
model     = VisionEncoderDecoderModel.from_pretrained(
    model_path,
    torch_dtype=torch.float16,
)
device = "cuda" if torch.cuda.is_available() else "cpu"
model.to(device)
model.eval()
print(f"[TrOCR] Model ready on {device}")

# ── Segment ───────────────────────────────────────────────────────────────────
image      = Image.open(image_path).convert("RGB")
line_crops = segment_lines(image)
print(f"[TrOCR] Detected {len(line_crops)} text lines.")

# ── Per-line OCR in batches ───────────────────────────────────────────────────
# Batch size 8 keeps VRAM usage predictable on a P100 / T4.
BATCH_SIZE = 8
all_lines  = []

for i in range(0, len(line_crops), BATCH_SIZE):
    batch        = line_crops[i : i + BATCH_SIZE]
    pixel_values = processor(images=batch, return_tensors="pt").pixel_values
    pixel_values = pixel_values.to(device, torch.float16)

    with torch.no_grad():
        gen_ids = model.generate(
            pixel_values,
            max_new_tokens=128,   # single text lines are short
        )

    texts = processor.batch_decode(gen_ids, skip_special_tokens=True)
    all_lines.extend(texts)
    print(f"  Lines {i+1}–{i+len(batch)}: {texts}")

out = "\n".join(all_lines)

# ── Cleanup ───────────────────────────────────────────────────────────────────
del model, pixel_values, gen_ids
torch.cuda.empty_cache()

# ── Write ─────────────────────────────────────────────────────────────────────
Path(output_file).write_text(out or "[TrOCR] No output generated.", encoding="utf-8")
print(f"[TrOCR] Done — {len(all_lines)} lines written.")




Writing /kaggle/working/run_trocr.py


## Gliner for NER

In [ ]:
%%writefile /kaggle/working/run_ner.py
# run_ner.py — Autonomous GLiNER NER Post-Processing Engine (ONNX)
# Runs inside: hf_env (Isolated Subprocess)

import sys
import json
import re
import warnings

warnings.filterwarnings("ignore")
from gliner import GLiNER

text_file = sys.argv[1]
use_case  = sys.argv[2]
out_file  = sys.argv[3]
# NEW: Accept the raw comma-separated string from Gradio
raw_labels_string = sys.argv[4] if len(sys.argv) > 4 else ""

# Parse and clean the labels inputted by the user
labels = [l.strip() for l in raw_labels_string.split(',') if l.strip()]

# ── 1. Read & Pre-process Text (Context Rebuilder) ────────────────────────────
def clean_and_rebuild_context(raw_text: str) -> str:
    """Fixes OCR fusions, merges disconnected PaddleOCR values, and standardizes text."""
    text = re.sub(r'\s*\[conf:[0-9.]+\]', '', raw_text)
    text = re.sub(r'<[^>]+>', ' ', text)
    text = text.replace('{', ' ').replace('}', ' ').replace('"', ' ')

    text = re.sub(r'(Tota[l1I]+)[.,: ]*(\d+)', r'Total: \2', text, flags=re.IGNORECASE)

    lines = [line.strip() for line in text.split('\n') if line.strip()]
    rebuilt_lines = []

    target_keys = ["total", "sub-total", "subtotal", "tax", "pb1", "service", "date", "amount"]

    i = 0
    while i < len(lines):
        current_line = lines[i]

        if re.fullmatch(r'\d{1,2}[xX%]', current_line, re.IGNORECASE) or current_line in ["X", "x", "1", "1x"]:
            i += 1
            continue

        is_target_key = any(key in current_line.lower() for key in target_keys)
        if is_target_key and i + 1 < len(lines):
            next_line = lines[i + 1]
            if any(char.isdigit() for char in next_line):
                rebuilt_lines.append(f"{current_line}: {next_line}")
                i += 2
                continue

        rebuilt_lines.append(current_line)
        i += 1

    text = "\n".join(rebuilt_lines)
    text = re.sub(r'([a-zA-Z]+)-\n([a-zA-Z]+)', r'\1\2', text)
    text = re.sub(r'[ \t]+', ' ', text)
    text = re.sub(r'\n\s*\n', '\n', text)

    return text.strip()

try:
    with open(text_file, 'r', encoding='utf-8') as f:
        raw_text = f.read()
except Exception as e:
    with open(out_file, 'w', encoding='utf-8') as f:
        json.dump({"error": f"Failed to read OCR text: {e}"}, f)
    sys.exit(1)

cleaned_text = clean_and_rebuild_context(raw_text)

if not cleaned_text or not labels:
    with open(out_file, 'w', encoding='utf-8') as f:
        json.dump({"Message": "No text extracted or no labels defined for NER."}, f)
    sys.exit(0)

# ── 2. Document Chunking (~400 words) ─────────────────────────────────────────
words = cleaned_text.split()
chunks = [" ".join(words[i : i + 400]) for i in range(0, len(words), 400)]

# ── 3. Runtime Model Loading (ONNX Backend) ───────────────────────────────────
print(f"[GLiNER] Loading ONNX model from onnx-community mirror...")
try:
    model = GLiNER.from_pretrained(
        "onnx-community/gliner_large-v2.1",
        load_onnx_model=True,
        onnx_model_file="onnx/model.onnx",
        load_tokenizer=True
    )
except Exception as e:
    with open(out_file, 'w', encoding='utf-8') as f:
        json.dump({"error": f"Failed to load GLiNER ONNX: {str(e)}"}, f)
    sys.exit(1)

# ── 4. Predict & Safely Merge (With Value Resolver) ───────────────────────────
print(f"[GLiNER] Running NER on {len(chunks)} chunk(s) for labels: {labels}")
merged_entities = {label: [] for label in labels}

kv_lookup = {}
for line in cleaned_text.split('\n'):
    if ':' in line:
        parts = line.split(':', 1)
        kv_lookup[parts[0].strip().lower()] = parts[1].strip()

for chunk in chunks:
    try:
        entities = model.predict_entities(chunk, labels, flat_ner=True, threshold=0.25)
        for ent in entities:
            lbl = ent["label"]
            txt = ent["text"].strip()

            if txt.lower() in kv_lookup:
                txt = kv_lookup[txt.lower()]
            elif ':' in txt:
                txt = txt.split(':', 1)[1].strip()

            if txt and txt not in merged_entities[lbl]:
                merged_entities[lbl].append(txt)
    except Exception as e:
        print(f"[GLiNER] Warning: Skipping chunk due to error: {e}")
        continue

final_output = {k: v[0] if len(v) == 1 else v for k, v in merged_entities.items() if v}

if not final_output:
    final_output = {"Message": "Not Found"}

# ── 5. Save Output ────────────────────────────────────────────────────────────
with open(out_file, 'w', encoding='utf-8') as f:
    json.dump(final_output, f, indent=2, ensure_ascii=False)

print(f"[GLiNER] NER extraction complete.")

Writing /kaggle/working/run_ner.py


In [ ]:
%%writefile /kaggle/working/run_router.py
# run_router.py — OpenAI CLIP Zero-Shot Fallback Classifier
# Runs inside: hf_env (Isolated Subprocess)

import sys
import torch
from pathlib import Path
from PIL import Image
import warnings

warnings.filterwarnings("ignore")
from transformers import CLIPProcessor, CLIPModel

image_path  = sys.argv[1]
output_file = sys.argv[2]

# ── Load Model (Zero-Shot Vision) ─────────────────────────────────────────────
model_id = "openai/clip-vit-base-patch32"
processor = CLIPProcessor.from_pretrained(model_id)
model = CLIPModel.from_pretrained(model_id).to("cuda:0", torch.float16)
model.eval()

# ── Prepare Layout Definitions ────────────────────────────────────────────────
image = Image.open(image_path).convert("RGB")

labels = [
    "a printed retail receipt from a store",
    "a structured business invoice with tables and totals",
    "messy cursive handwritten text on paper",
    "a formal scanned legal document or contract"
]
category_map = ["Receipt", "Invoice", "Handwriting", "Legal PDF"]

# ── Inference ─────────────────────────────────────────────────────────────────
inputs = processor(text=labels, images=image, return_tensors="pt", padding=True)
inputs = {k: v.to("cuda:0") for k, v in inputs.items()}
inputs["pixel_values"] = inputs["pixel_values"].to(torch.float16)

with torch.no_grad():
    outputs = model(**inputs)
    probs = outputs.logits_per_image.softmax(dim=1)[0]

best_idx   = probs.argmax().item()
confidence = probs[best_idx].item()
prediction = category_map[best_idx]

# ── VRAM Cleanup & Output ─────────────────────────────────────────────────────
del model, inputs
torch.cuda.empty_cache()

Path(output_file).write_text(f"{prediction}|{confidence:.4f}", encoding="utf-8")

Writing /kaggle/working/run_router.py


## UI Design

In [ ]:
%%writefile /kaggle/working/app.py
import subprocess
import time
from pathlib import Path
from PIL import Image, ExifTags


import gradio as gr

print(f"✅ Gradio version: {gr.__version__}")

# ── 1. Configuration & Paths ──────────────────────────────────────────────────
BASE = "/kaggle/input/datasets/abhishekswami10/document-ai-offline-models/document-ai-models"
WORK_DIR    = Path("/kaggle/working")
OUTPUT_FILE = WORK_DIR / "model_output.txt"
ROUTER_FILE = WORK_DIR / "router_output.txt"
NER_FILE    = WORK_DIR / "ner_output.json"

PADDLE_PY = "/kaggle/working/paddle_env/bin/python"
DOCTR_PY  = "/kaggle/working/doctr_env/bin/python"
HF_PY     = "/kaggle/working/hf_env/bin/python"

subprocess.run("pip install --quiet Pillow PyMuPDF", shell=True, check=True)
subprocess.run(f"{HF_PY} -m pip install --quiet gliner onnxruntime-gpu", shell=True)

MODEL_PATHS = {
    "qwen"     : f"{BASE}/qwen2.5-vl-7b-awq",
    "donut"    : "naver-clova-ix/donut-base-finetuned-cord-v2",
    "florence" : f"{BASE}/florence-2-large",
    "trocr"    : f"{BASE}/trocr-handwritten",
    "paddle"   : None,
    "doctr"    : None,
}

USE_CASE_MODEL_MAP = {
    "Receipt"     : ["PaddleOCR", "Qwen2.5-VL-7B-AWQ", "Donut", "Florence-2-Large"],
    "Invoice"     : ["Qwen2.5-VL-7B-AWQ", "Donut", "Florence-2-Large", "PaddleOCR"],
    "Handwriting" : ["TrOCR-Large-Handwritten", "Qwen2.5-VL-7B-AWQ"],
    "Legal PDF"   : ["DocTR", "Qwen2.5-VL-7B-AWQ", "PaddleOCR", "Florence-2-Large"],
}

QWEN_PROMPTS = {
    "Receipt"    : "Extract ALL text from this retail receipt exactly as it appears. Preserve item names, prices, quantities, totals, date, and store info. Format as clean plain text.",
    "Invoice"    : "Extract all fields from this invoice and return structured JSON: vendor, invoice_number, date, line_items (description, qty, unit_price, amount), subtotal, tax, total.",
    "Handwriting": "Carefully transcribe ALL handwritten text exactly as written. Preserve line breaks and punctuation.",
    "Legal PDF"  : "Extract ALL text from this scanned legal document exactly as it appears. Preserve headings, numbered sections, paragraph structure, and any visible signatures or stamps.",
}

# NEW: Default labels mapped to Use Cases for the UI
DEFAULT_LABELS = {
    "Receipt"    : "Store, Date, Phone, Tax, Total",
    "Invoice"    : "Vendor, Invoice Number, Due Date, GST Number, Grand Total",
    "Legal PDF"  : "Plaintiff, Defendant, Filing Date, Court Name, Jurisdiction, Settlement Amount",
    "Handwriting": "Person Name, Date, Location"
}

MODEL_SPECS = {
    "Qwen2.5-VL-7B-AWQ": """
**⚙️ Production Hardware Requirements**
* **VRAM Required:** 5–6 GB (AWQ INT4)
* **Speed (1 page):** 25–55 sec
* **Minimum GPU:** NVIDIA T4 15GB or RTX 4090 24GB
* **Recommended GPU:** NVIDIA L4 24GB (~$0.35–$0.70/hr)
* **Concurrency:** 15-25 requests (on 24GB GPU via vLLM)
* **Best For:** Complex mixed docs, JSON extraction, degraded images
""",
    "Donut": """
**⚙️ Production Hardware Requirements**
* **VRAM Required:** ~1.5 GB
* **Speed (1 page):** 8–20 sec
* **Minimum GPU:** NVIDIA RTX 3060 12GB or T4
* **Recommended GPU:** NVIDIA T4 15GB (~$0.52/hr)
* **Concurrency:** 5-8 requests (on T4 15GB)
* **Best For:** Fast structured receipt / invoice JSON
""",
    "Florence-2-Large": """
**⚙️ Production Hardware Requirements**
* **VRAM Required:** ~3 GB
* **Speed (1 page):** 15–25 sec
* **Minimum GPU:** NVIDIA RTX 3070 8GB or T4
* **Recommended GPU:** NVIDIA T4 15GB (~$0.52/hr)
* **Concurrency:** 4-12 requests (on T4 15GB)
* **Best For:** Spatial invoice layout, bounding box OCR
""",
    "TrOCR-Large-Handwritten": """
**⚙️ Production Hardware Requirements**
* **VRAM Required:** ~2.5 GB
* **Speed (1 page):** 10–25 sec
* **Minimum GPU:** NVIDIA GTX 1060 6GB or CPU Server
* **Recommended GPU:** NVIDIA T4 15GB (~$0.52/hr)
* **Concurrency:** 5 concurrent GPU instances, or CPU batching
* **Best For:** Handwriting, cursive text, single-line accuracy
""",
    "DocTR": """
**⚙️ Production Hardware Requirements**
* **VRAM Required:** ~0.8 GB
* **Speed (1 page):** 8–18 sec
* **Minimum GPU:** CPU Server or any GPU
* **Recommended Hardware:** Compute-Optimized CPU (~$0.10/hr)
* **Concurrency:** 17+ concurrent requests on a 15GB GPU
* **Best For:** Legal PDFs, multi-column layouts, high volume
""",
    "PaddleOCR": """
**⚙️ Production Hardware Requirements**
* **VRAM Required:** ~0.3 GB
* **Speed (1 page):** 5–10 sec
* **Minimum GPU:** CPU Server
* **Recommended Hardware:** Standard CPU EC2 Instance (~$0.05/hr)
* **Concurrency:** 45+ concurrent instances theoretically
* **Best For:** High-speed printed text, budget deployments
"""
}
ROUTER = {
    "PaddleOCR"              : (PADDLE_PY, "/kaggle/working/run_paddle.py",   []),
    "Qwen2.5-VL-7B-AWQ"      : (HF_PY,     "/kaggle/working/run_qwen.py",     [MODEL_PATHS["qwen"]]),
    "Donut"                  : (HF_PY,     "/kaggle/working/run_donut.py",    [MODEL_PATHS["donut"]]),
    "Florence-2-Large"       : (HF_PY,     "/kaggle/working/run_florence.py", [MODEL_PATHS["florence"]]),
    "TrOCR-Large-Handwritten": (HF_PY,     "/kaggle/working/run_trocr.py",    [MODEL_PATHS["trocr"]]),
    "DocTR"                  : (DOCTR_PY,  "/kaggle/working/run_doctr.py",    []),
}

# ── 3. LLD: Metadata Heuristic Engine ─────────────────────────────────────────
def check_metadata_heuristic(file_path: str):
    try:
        ext = Path(file_path).suffix.lower()
        if ext == '.pdf':
            doc = fitz.open(file_path)
            metadata = doc.metadata or {}
            producer = metadata.get("producer", "").lower()
            creator  = metadata.get("creator", "").lower()
            doc.close()
            if "quickbooks" in producer or "stripe" in creator or "invoice" in creator:
                return "Invoice"
            if "docusign" in producer or "adobe sign" in producer:
                return "Legal PDF"
        elif ext in ['.jpg', '.jpeg', '.png', '.tiff']:
            img = Image.open(file_path)
            exif = img.getexif()
            if exif:
                for k, v in exif.items():
                    tag = ExifTags.TAGS.get(k, k)
                    if tag == "Software":
                        software = str(v).lower()
                        if "camscanner" in software or "adobe scan" in software:
                            return "Legal PDF"
    except Exception:
        pass
    return None

def pdf_to_jpeg(pdf_path: str) -> str:
    doc  = fitz.open(pdf_path)
    page = doc[0]
    mat  = fitz.Matrix(200 / 72, 200 / 72)
    pix  = page.get_pixmap(matrix=mat, colorspace=fitz.csRGB)
    out  = str(WORK_DIR / "pdf_page0.jpg")
    pix.save(out)
    doc.close()
    return out

# ── 5. HLD: The Triage Controller ─────────────────────────────────────────────
def auto_detect_document(image_path, pdf_path):
    if not image_path and not pdf_path:
        return gr.update(), gr.update(), gr.update(), gr.update(), "⚠️ No document uploaded.", gr.update()

    original_file = image_path if image_path else pdf_path
    image_input = original_file

    t0 = time.time()
    predicted_category = check_metadata_heuristic(original_file)

    if predicted_category:
        elapsed = time.time() - t0
        status = f"⚡ Fast-Track (Metadata): {predicted_category} (100.0%) in {elapsed:.3f}s"
    else:
        ext = Path(original_file).suffix.lower()
        if ext == ".pdf":
            try:
                image_input = pdf_to_jpeg(original_file)
            except Exception as e:
                return gr.update(), gr.update(), gr.update(), gr.update(), f"❌ PDF conversion failed: {e}", gr.update()

        ROUTER_FILE.write_text("", encoding="utf-8")
        cmd = [HF_PY, "/kaggle/working/run_router.py", image_input, str(ROUTER_FILE)]

        t1 = time.time()
        try:
            subprocess.run(cmd, capture_output=False, timeout=60)
            elapsed = time.time() - t1
            output = ROUTER_FILE.read_text(encoding="utf-8").strip()
            if "|" in output:
                predicted_category, conf = output.split("|")
                status = f"🧠 AI Fallback (CLIP): {predicted_category} ({float(conf)*100:.1f}%) in {elapsed:.1f}s"
            else:
                return gr.update(), gr.update(), gr.update(), gr.update(), "⚠️ CLIP Classification failed.", gr.update()
        except Exception as e:
            return gr.update(), gr.update(), gr.update(), gr.update(), f"❌ Fallback Error: {e}", gr.update()

    models = USE_CASE_MODEL_MAP.get(predicted_category, [])
    default_model = models[0] if models else None
    is_qwen = (default_model == "Qwen2.5-VL-7B-AWQ")

    # Cascade updates including the new NER Labels
    return (
        gr.update(value=predicted_category),
        gr.update(choices=models, value=default_model),
        gr.update(visible=is_qwen, value=QWEN_PROMPTS.get(predicted_category, "") if is_qwen else ""),
        gr.update(value=MODEL_SPECS.get(default_model, "")),
        status,
        gr.update(value=DEFAULT_LABELS.get(predicted_category, ""))
    )

def handle_use_case_change(use_case: str):
    models = USE_CASE_MODEL_MAP.get(use_case, [])
    default_model = models[0] if models else None
    is_qwen = (default_model == "Qwen2.5-VL-7B-AWQ")
    return (
        gr.update(choices=models, value=default_model),
        gr.update(visible=is_qwen, value=QWEN_PROMPTS.get(use_case, "") if is_qwen else ""),
        gr.update(value=MODEL_SPECS.get(default_model, "")),
        gr.update(value=DEFAULT_LABELS.get(use_case, "")) # NEW: Update Labels on Use Case change
    )

def handle_model_change(use_case: str, model_name: str):
    is_qwen = (model_name == "Qwen2.5-VL-7B-AWQ")
    return (
        gr.update(visible=is_qwen, value=QWEN_PROMPTS.get(use_case, "") if is_qwen else ""),
        gr.update(value=MODEL_SPECS.get(model_name, ""))
    )

# CHANGED: Refactored into a Yielding Generator for Streaming UX
def process_document(image_path, pdf_path, use_case: str, model_name: str, custom_prompt: str, custom_labels: str):
    if not image_path and not pdf_path:
        yield "", "", "❌ Please upload an image or PDF."
        return

    input_path = image_path if image_path else pdf_path
    OUTPUT_FILE.write_text("", encoding="utf-8")
    ext = Path(input_path).suffix.lower()
    if ext == ".pdf": input_path = pdf_to_jpeg(input_path)

    python_bin, script_path, extra_args = ROUTER[model_name]
    cmd = [python_bin, script_path, input_path, use_case, str(OUTPUT_FILE), *extra_args]
    if model_name == "Qwen2.5-VL-7B-AWQ": cmd.append(custom_prompt)

    # --- 1. Run OCR / VLM ---
    t_start = time.time()
    try:
        result = subprocess.run(cmd, capture_output=False, timeout=360)
        elapsed = time.time() - t_start
        status = f"✅ {model_name} · {elapsed:.1f}s" if result.returncode == 0 else f"⚠️ {model_name} exited code {result.returncode}"
    except Exception as exc:
        yield "", "", f"❌ Error: {exc}"
        return

    output_text = OUTPUT_FILE.read_text(encoding="utf-8").strip() or f"[{model_name}] No output extracted."

    # --- UI YIELD 1: Push Raw OCR to UI instantly ---
    yield output_text, "⏳ Reading text context to extract Entities...", status + " | ⏳ Running NER Engine..."

    # --- 2. Run GLiNER NER Post-Processing ---
    NER_FILE.write_text("", encoding="utf-8")
    ner_cmd = [HF_PY, "/kaggle/working/run_ner.py", str(OUTPUT_FILE), use_case, str(NER_FILE), custom_labels]

    print(f"\n{'='*60}\n  🧠  NER Engine : GLiNER (ONNX)\n{'='*60}")
    try:
        subprocess.run(ner_cmd, capture_output=False, timeout=120)
        ner_json = NER_FILE.read_text(encoding="utf-8").strip()
    except Exception as e:
        ner_json = f'{{"error": "NER execution failed: {e}"}}'

    # --- UI YIELD 2: Push Final NER JSON to UI ---
    yield output_text, ner_json, status + " | ✅ NER Complete"

# ── 7. Presentation Layer (Gradio) ────────────────────────────────────────────
with gr.Blocks(title="Document AI & VLM Benchmarking Pipeline", theme=gr.themes.Soft()) as demo:
    gr.Markdown("# 📄 Document AI & VLM Benchmarking Pipeline\n**Enterprise Hybrid-Routing Architecture with GLiNER NER**")

    with gr.Row():
        with gr.Column(scale=4):
            image_input = gr.Image(label="Upload Image", type="filepath")
            pdf_input   = gr.File(label="Upload PDF", file_types=[".pdf"], type="filepath")

            with gr.Row():
                use_case_dd = gr.Dropdown(label="📂 Step 1 — Document Type", choices=list(USE_CASE_MODEL_MAP.keys()), value="Receipt", scale=3)
                detect_btn  = gr.Button("🔍 Auto-Detect", variant="secondary", scale=1)

            _default_model = USE_CASE_MODEL_MAP["Receipt"][0]
            model_dd    = gr.Dropdown(label="🤖 Step 2 — Select Model", choices=USE_CASE_MODEL_MAP["Receipt"], value=_default_model)

            # NEW: The Input box for custom NER Labels
            ner_labels_box = gr.Textbox(label="🏷️ Custom NER Labels (Comma-separated)", lines=2, value=DEFAULT_LABELS.get("Receipt", ""))
            prompt_box  = gr.Textbox(label="📝 Custom Qwen Prompt", lines=3, visible=False)

            process_btn = gr.Button("⚡ Process Document", variant="primary")

        with gr.Column(scale=5):
            status_box  = gr.Textbox(label="🔄 Pipeline Status", interactive=False, lines=2)
            output_box  = gr.Textbox(label="📝 Raw Extracted Text / Layout", interactive=False, lines=12)
            ner_box     = gr.Textbox(label="🧠 Structured Entities (GLiNER ONNX)", interactive=False, lines=8)
            specs_panel = gr.Markdown(value=MODEL_SPECS.get(_default_model, ""))

    # Wiring
    inputs_to_route = [image_input, pdf_input]
    outputs_from_route = [use_case_dd, model_dd, prompt_box, specs_panel, status_box, ner_labels_box]

    image_input.upload(fn=auto_detect_document, inputs=inputs_to_route, outputs=outputs_from_route)
    pdf_input.upload(fn=auto_detect_document, inputs=inputs_to_route, outputs=outputs_from_route)
    detect_btn.click(fn=auto_detect_document, inputs=inputs_to_route, outputs=outputs_from_route)

    use_case_dd.change(fn=handle_use_case_change, inputs=use_case_dd, outputs=[model_dd, prompt_box, specs_panel, ner_labels_box])
    model_dd.change(fn=handle_model_change, inputs=[use_case_dd, model_dd], outputs=[prompt_box, specs_panel])

    process_btn.click(fn=process_document, inputs=[image_input, pdf_input, use_case_dd, model_dd, prompt_box, ner_labels_box], outputs=[output_box, ner_box, status_box])

if __name__ == "__main__":
    demo.launch(server_name="0.0.0.0", share=True, show_error=True)

Overwriting /kaggle/working/app.py


In [ ]:
!python /kaggle/working/app.py

✅ Gradio version: 5.50.0
/kaggle/working/app.py:274: DeprecationWarning: The 'theme' parameter in the Blocks constructor will be removed in Gradio 6.0. You will need to pass 'theme' to Blocks.launch() instead.
  with gr.Blocks(title="Document AI & VLM Benchmarking Pipeline", theme=gr.themes.Soft()) as demo:
* Running on local URL:  http://0.0.0.0:7860
* Running on public URL: https://8aaec50dfff1ff295e.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
2026-03-19 04:55:05.049333: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1773896105.070856     323 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been